# Sistem Prediksi Harga Saham PT Telekomunikasi Indonesia Tbk (TLKM.JK)
**Time-Series Regression (Ridge Regression, OLS) & Holt's Linear Trend dengan Evaluasi Naive Baseline**

Notebook ini menyajikan alur kerja lengkap mulai dari *Data Understanding*, *EDA*, *Preprocessing*, *Feature Engineering*, *Modeling*, *Evaluasi*, hingga ekspor `data/tlkm_clean.csv` dan pengecekan konsistensi terhadap modul `forecaster.py`.

## 1. Import Library & Konfigurasi

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 20)
print('Library siap digunakan.')

## 2. Data Understanding
### 2.1 Muat atau Unggah Dataset CSV
Jika dijalankan di **Google Colab**, sel ini menampilkan tombol unggah file CSV (`SAHAM - PT Telekomunikasi Indonesia Tbk (TLKM.JK) - Sheet1.csv`). Jika dijalankan secara lokal, sel akan mencari file CSV secara otomatis di folder proyek.

In [ ]:
RAW_FILENAMES = [
    'SAHAM - PT Telekomunikasi Indonesia Tbk (TLKM.JK) - Sheet1.csv',
    '../SAHAM - PT Telekomunikasi Indonesia Tbk (TLKM.JK) - Sheet1.csv',
    'data/tlkm_clean.csv',
    '../data/tlkm_clean.csv',
]

IN_COLAB = False
try:
    from google.colab import files  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

csv_path = None
for candidate in RAW_FILENAMES:
    if Path(candidate).exists():
        csv_path = Path(candidate)
        break

if csv_path is None and IN_COLAB:
    print('Unggah file CSV dataset saham TLKM.JK:')
    uploaded = files.upload()
    csv_path = Path(next(iter(uploaded.keys())))
elif csv_path is None:
    raise FileNotFoundError(
        'File CSV tidak ditemukan. Letakkan SAHAM - PT Telekomunikasi Indonesia Tbk (TLKM.JK) - Sheet1.csv '
        'di folder proyek atau folder notebook/.'
    )

df_raw = pd.read_csv(csv_path)
print(f'File dimuat: {csv_path} | Bentuk data: {df_raw.shape}')
df_raw.head()

In [ ]:
df_raw.info()
print('\nJumlah nilai kosong per kolom:')
print(df_raw.isna().sum())

## 3. Exploratory Data Analysis (EDA) & Pembersihan Data (Preprocessing)
Pada dataset mentah ditemukan dua hal penting:
1. Kolom `Date` berformat `DD/MM/YYYY` dan kolom `Volume` memakai pemisah ribuan titik (`125.291.200`).
2. Terdapat 1 baris anomali `Volume = 0` pada tanggal `13/03/2020` (OHLC flat di `3310.0`). Nilai ini diimputasi menggunakan median 5 hari perdagangan sebelumnya agar tidak mengganggu fitur log-volume.

In [ ]:
def clean_stock_data_nb(df_in: pd.DataFrame):
    df = df_in.copy()
    df.columns = [str(c).strip() for c in df.columns]
    parsed = pd.to_datetime(df['Date'], format='%d/%m/%Y', errors='coerce')
    if parsed.isna().sum() > len(df) * 0.5:
        parsed = pd.to_datetime(df['Date'], format='%Y-%m-%d', errors='coerce')
    df['Date'] = parsed.fillna(pd.to_datetime(df['Date'], dayfirst=True, errors='coerce'))

    for col in ['Open', 'High', 'Low', 'Close', 'Adj Close']:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    if not pd.api.types.is_numeric_dtype(df['Volume']):
        df['Volume'] = pd.to_numeric(
            df['Volume'].astype(str).str.strip().str.replace('.', '', regex=False).str.replace(',', '', regex=False),
            errors='coerce',
        )

    df = df.dropna(subset=['Date', 'Open', 'High', 'Low', 'Close', 'Adj Close'])
    df = df.sort_values('Date').drop_duplicates(subset=['Date'], keep='last').reset_index(drop=True)
    df = df[df['High'] >= df['Low']].reset_index(drop=True)

    zero_mask = (df['Volume'].isna()) | (df['Volume'] <= 0)
    zero_dates = df.loc[zero_mask, 'Date'].dt.strftime('%Y-%m-%d').tolist()
    if zero_mask.any():
        df.loc[zero_mask, 'Volume'] = np.nan
        df['Volume'] = df['Volume'].fillna(df['Volume'].rolling(5, min_periods=1).median()).fillna(df['Volume'].median())

    df['Volume'] = df['Volume'].round().astype('int64')
    df['Date'] = df['Date'].dt.strftime('%Y-%m-%d')
    cols = ['Date', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']
    return df[cols], zero_dates

df_clean, zero_dates = clean_stock_data_nb(df_raw)
print(f'Baris bersih: {len(df_clean)} | Rentang: {df_clean["Date"].iloc[0]} s.d. {df_clean["Date"].iloc[-1]}')
print(f'Anomali Volume=0 diimputasi: {zero_dates}')
df_clean.describe().round(2)

In [ ]:
dates_dt = pd.to_datetime(df_clean['Date'])
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 5.5), sharex=True, gridspec_kw={'height_ratios': [2.2, 1]})
ax1.plot(dates_dt, df_clean['Close'], label='Close', color='#1E3A5F', linewidth=1.4)
ax1.plot(dates_dt, df_clean['Adj Close'], label='Adj Close', color='#64748B', linestyle='--', linewidth=1.1)
ax1.set_title('Riwayat Harga Penutupan & Volume Perdagangan TLKM.JK (2019–2024)', fontweight='bold')
ax1.set_ylabel('Harga (Rp)')
ax1.legend()

ax2.bar(dates_dt, df_clean['Volume'] / 1e6, color='#B91C1C', alpha=0.65, width=2.0)
ax2.set_ylabel('Volume (Juta)')
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout()
plt.show()

## 4. Feature Engineering & Data Preparation
Seluruh fitur pada hari *t* hanya menggunakan informasi hingga hari *t* (tanpa *look-ahead bias*).
Target prediksi `Target_Close` adalah harga penutupan hari bursa berikutnya (*t* + 1). Pembagian data latih dan uji dilakukan **secara kronologis (80% / 20%) tanpa pengacakan**.

In [ ]:
FEATURE_COLUMNS = [
    'Close', 'Open', 'High', 'Low',
    'Lag_1', 'Lag_2', 'Lag_3', 'Lag_5',
    'MA_5', 'MA_10', 'MA_20',
    'EMA_12', 'EMA_26', 'MACD', 'RSI_14',
    'Volatility_10', 'Return_1d', 'Return_5d',
    'HL_Range_Pct', 'OC_Change_Pct', 'Volume_Log',
]

def compute_rsi(close: pd.Series, window: int = 14) -> pd.Series:
    delta = close.diff()
    gain = delta.clip(lower=0.0)
    loss = -delta.clip(upper=0.0)
    avg_gain = gain.ewm(alpha=1.0 / window, min_periods=window, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1.0 / window, min_periods=window, adjust=False).mean()
    rs = avg_gain / avg_loss.replace(0.0, np.nan)
    return (100.0 - (100.0 / (1.0 + rs))).fillna(50.0)

def build_features_nb(df_in: pd.DataFrame) -> pd.DataFrame:
    df = df_in.copy()
    df['Date'] = pd.to_datetime(df['Date'])
    close = df['Close'].astype(float)
    df['Lag_1'] = close.shift(1)
    df['Lag_2'] = close.shift(2)
    df['Lag_3'] = close.shift(3)
    df['Lag_5'] = close.shift(5)
    df['MA_5'] = close.rolling(5).mean()
    df['MA_10'] = close.rolling(10).mean()
    df['MA_20'] = close.rolling(20).mean()
    df['EMA_12'] = close.ewm(span=12, adjust=False).mean()
    df['EMA_26'] = close.ewm(span=26, adjust=False).mean()
    df['MACD'] = df['EMA_12'] - df['EMA_26']
    df['RSI_14'] = compute_rsi(close, 14)
    df['Return_1d'] = close.pct_change(1)
    df['Return_5d'] = close.pct_change(5)
    df['Volatility_10'] = df['Return_1d'].rolling(10).std()
    df['HL_Range_Pct'] = (df['High'] - df['Low']) / close
    df['OC_Change_Pct'] = (df['Close'] - df['Open']) / df['Open']
    df['Volume_Log'] = np.log1p(df['Volume'].astype(float))
    df['Target_Close'] = close.shift(-1)
    df['Target_Date'] = df['Date'].shift(-1)
    return df.dropna(subset=FEATURE_COLUMNS + ['Target_Close']).reset_index(drop=True)

feat_df = build_features_nb(df_clean)
split_idx = int(len(feat_df) * 0.8)
train_df, test_df = feat_df.iloc[:split_idx], feat_df.iloc[split_idx:].reset_index(drop=True)
print(f'Data latih: {len(train_df)} baris | Data uji: {len(test_df)} baris')

## 5 & 6. Pemodelan dan Evaluasi
Kita membandingkan 4 pendekatan:
- **Ridge Regression** (`alpha=10.0`)
- **Linear Regression (OLS)**
- **Holt's Linear Trend** (Double Exponential Smoothing)
- **Naive Baseline** (*Persistence*: $\hat{y}_{t+1} = y_t$)

In [ ]:
def fit_ridge_nb(X_tr, y_tr, alpha=10.0):
    mu = X_tr.mean(axis=0)
    sd = np.where(X_tr.std(axis=0) < 1e-12, 1.0, X_tr.std(axis=0))
    Xs = (X_tr - mu) / sd
    ym = float(y_tr.mean())
    reg = (alpha if alpha > 0 else 1e-6) * np.eye(Xs.shape[1])
    w = np.linalg.solve(Xs.T @ Xs + reg, Xs.T @ (y_tr - ym))
    return mu, sd, w, ym

def eval_metrics_nb(y_true, y_pred, prev_close):
    err = y_true - y_pred
    mae = float(np.mean(np.abs(err)))
    rmse = float(np.sqrt(np.mean(err ** 2)))
    mape = float(np.mean(np.abs(err / y_true)) * 100.0)
    r2 = float(1.0 - np.sum(err ** 2) / np.sum((y_true - np.mean(y_true)) ** 2))
    da = float(np.mean(np.sign(y_true - prev_close) == np.sign(y_pred - prev_close)) * 100.0)
    return {'MAE': round(mae, 2), 'RMSE': round(rmse, 2), 'MAPE (%)': round(mape, 2), 'R2': round(r2, 4), 'Directional Accuracy (%)': round(da, 2)}

X_tr = train_df[FEATURE_COLUMNS].to_numpy(float)
y_tr = train_df['Target_Close'].to_numpy(float)
X_te = test_df[FEATURE_COLUMNS].to_numpy(float)
y_te = test_df['Target_Close'].to_numpy(float)
prev_te = test_df['Close'].to_numpy(float)

mu_r, sd_r, w_r, ym_r = fit_ridge_nb(X_tr, y_tr, alpha=10.0)
pred_ridge = ((X_te - mu_r) / sd_r) @ w_r + ym_r

mu_o, sd_o, w_o, ym_o = fit_ridge_nb(X_tr, y_tr, alpha=0.0)
pred_ols = ((X_te - mu_o) / sd_o) @ w_o + ym_o

pred_naive = prev_te.copy()

results = []
for name, preds in [('Ridge Regression', pred_ridge), ('Linear Regression (OLS)', pred_ols), ('Naive Baseline', pred_naive)]:
    row = {'Model': name}
    row.update(eval_metrics_nb(y_te, preds, prev_te))
    results.append(row)

eval_table_nb = pd.DataFrame(results).sort_values('RMSE').reset_index(drop=True)
eval_table_nb

## 7. Visualisasi Aktual vs Prediksi pada Data Uji

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.3))
ax.plot(test_df['Target_Date'], y_te, label='Aktual (Close)', color='#1E3A5F', linewidth=1.6)
ax.plot(test_df['Target_Date'], pred_ridge, label='Prediksi Ridge Regression', color='#B91C1C', linestyle='--', linewidth=1.4)
ax.set_title('Perbandingan Harga Aktual vs Prediksi Ridge Regression (Data Uji 20%)', fontweight='bold')
ax.set_ylabel('Harga (Rp)')
ax.legend()
plt.tight_layout()
plt.show()

## 8. Ekspor Data Bersih (`data/tlkm_clean.csv`)
Di lingkungan lokal, file disimpan ke `../data/tlkm_clean.csv` (atau `data/tlkm_clean.csv`). Di Google Colab, file akan otomatis diunduh ke komputer Anda.

In [ ]:
out_path = Path('../data/tlkm_clean.csv') if Path('../forecaster.py').exists() else Path('data/tlkm_clean.csv')
out_path.parent.mkdir(parents=True, exist_ok=True)
df_clean.to_csv(out_path, index=False)
print(f'Data bersih tersimpan di: {out_path}')

if IN_COLAB:
    files.download(str(out_path))

## 9. Cek Konsistensi dengan `forecaster.py`
Bagian ini hanya berjalan apabila notebook dibuka secara lokal dari folder `notebook/` (sejajar dengan `forecaster.py`). Di Google Colab, bagian ini otomatis dilewati.

In [ ]:
import sys
parent_dir = Path('..').resolve()
if (parent_dir / 'forecaster.py').exists():
    if str(parent_dir) not in sys.path:
        sys.path.insert(0, str(parent_dir))
    from forecaster import evaluate_models

    summary_app, _ = evaluate_models(df_clean, train_ratio=0.8, ridge_alpha=10.0)
    ridge_app_rmse = float(summary_app.loc[summary_app['Model'] == 'Ridge Regression', 'RMSE'].iloc[0])
    ridge_nb_rmse = float(eval_table_nb.loc[eval_table_nb['Model'] == 'Ridge Regression', 'RMSE'].iloc[0])
    assert abs(ridge_app_rmse - ridge_nb_rmse) < 1e-6, f'Mismatch RMSE: {ridge_app_rmse} vs {ridge_nb_rmse}'
    print('Notebook dan aplikasi konsisten.')
else:
    print('Lewati cek konsistensi (forecaster.py tidak ditemukan di parent directory — normal di Google Colab).')